# Wind Power Reliability Analysis

Analysis of UK national wind generation (Jan 2025 onwards) to determine how reliably wind can meet electricity demand. Recommends firm capacity (MW) based on exceedance probability, peak vs off-peak, and drought events.

In [1]:
import requests
import json
import warnings

In [2]:
import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import gaussian_kde

In [3]:
warnings.filterwarnings("ignore")

In [4]:
THEME = {
    "figure.dpi":       130,
    "figure.facecolor": "#0d1117",
    "axes.facecolor":   "#161b22",
    "axes.edgecolor":   "#30363d",
    "axes.labelcolor":  "#c9d1d9",
    "xtick.color":      "#8b949e",
    "ytick.color":      "#8b949e",
    "text.color":       "#c9d1d9",
    "grid.color":       "#21262d",
    "grid.linewidth":   0.8,
    "legend.facecolor": "#161b22",
    "legend.edgecolor": "#30363d",
    "font.family":      "sans-serif",
}
plt.rcParams.update(THEME)

In [5]:
ELEXON_BASE  = "https://data.elexon.co.uk/bmrs/api/v1"
FETCH_START  = "2025-01-01T00:00:00Z"
FETCH_END    = pd.Timestamp.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
PEAK_HOURS   = [16, 17, 18, 19, 20]         
INSTALLED_MW = 29_000                  

In [6]:
print(f"Analysis window : {FETCH_START}  →  {FETCH_END}")
print(f"Peak hours (UTC): {PEAK_HOURS[0]:02d}:00 – {PEAK_HOURS[-1]:02d}:30")
print(f"Installed cap.  : {INSTALLED_MW:,} MW  (used for capacity factor only)")

Analysis window : 2025-01-01T00:00:00Z  →  2026-03-18T09:56:37Z
Peak hours (UTC): 16:00 – 20:30
Installed cap.  : 29,000 MW  (used for capacity factor only)


In [7]:
def fetch_elexon(endpoint: str, params: dict) -> list[dict]:
    url = f"{ELEXON_BASE}{endpoint}"
    with requests.get(url, params=params, stream=True, timeout=120) as r:
        r.raise_for_status()
        text = r.text.strip()
    if text.startswith("["):
        return json.loads(text)
    records = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return records

In [8]:
print("Fetching actual wind generation (FUELHH WIND) …")
raw = fetch_elexon("/datasets/FUELHH/stream", {
    "from":     FETCH_START,
    "to":       FETCH_END,
    "fuelType": "WIND",
    "format":   "json",
})
print(f"Raw records: {len(raw):,}")

Fetching actual wind generation (FUELHH WIND) …


Raw records: 19


In [9]:
df = (
    pd.DataFrame(raw)
    .query("fuelType == 'WIND'")
    [["startTime", "generation"]]
    .copy()
)
df["startTime"]  = pd.to_datetime(df["startTime"], utc=True)
df["generation"] = pd.to_numeric(df["generation"], errors="coerce")
df = (df.dropna(subset=["generation"])
       .sort_values("startTime")
       .drop_duplicates("startTime")
       .reset_index(drop=True))

In [10]:
n_expected = len(pd.date_range(
    df["startTime"].min(), df["startTime"].max(), freq="30min"))
n_missing  = n_expected - len(df)

In [11]:
print(f"Clean records  : {len(df):,}")
print(f"Date range     : {df['startTime'].min()}  →  {df['startTime'].max()}")
print(f"Missing slots  : {n_missing:,} / {n_expected:,}  "
      f"({n_missing/n_expected*100:.1f}% gap)")
if len(df) < 10:
    raise SystemExit("Insufficient data (need ≥10 records). Elexon API may return limited data.")
df.head(3)

Clean records  : 19
Date range     : 2026-03-18 00:00:00+00:00  →  2026-03-18 09:00:00+00:00
Missing slots  : 0 / 19  (0.0% gap)


,startTime,generation
0,2026-03-18 00:00:00+00:00,10248
1,2026-03-18 00:30:00+00:00,9514
2,2026-03-18 01:00:00+00:00,8928


In [12]:
gen = df["generation"]

In [13]:
print("=" * 52)
print("   ACTUAL WIND GENERATION — DESCRIPTIVE STATISTICS")
print("=" * 52)
rows = [
    ("N (30-min slots)",     f"{len(gen):,}"),
    ("Days of data",         f"{len(gen)/48:.1f}"),
    ("Min",                  f"{gen.min():>10,.0f} MW"),
    ("P1",                   f"{gen.quantile(0.01):>10,.0f} MW"),
    ("P5",                   f"{gen.quantile(0.05):>10,.0f} MW"),
    ("P10  ← firm capacity", f"{gen.quantile(0.10):>10,.0f} MW"),
    ("P20",                  f"{gen.quantile(0.20):>10,.0f} MW"),
    ("P25",                  f"{gen.quantile(0.25):>10,.0f} MW"),
    ("Median (P50)",         f"{gen.quantile(0.50):>10,.0f} MW"),
    ("Mean",                 f"{gen.mean():>10,.0f} MW"),
    ("P75",                  f"{gen.quantile(0.75):>10,.0f} MW"),
    ("P90",                  f"{gen.quantile(0.90):>10,.0f} MW"),
    ("P95",                  f"{gen.quantile(0.95):>10,.0f} MW"),
    ("Max",                  f"{gen.max():>10,.0f} MW"),
    ("Std Dev",              f"{gen.std():>10,.0f} MW"),
    ("Coeff of Variation",   f"{gen.std()/gen.mean()*100:>9.1f}%"),
]
for lbl, val in rows:
    print(f"   {lbl:<25} {val}")
print("=" * 52)
print(f"\n→ Wind < 500 MW  : {(gen < 500).mean()*100:.1f}% of all slots")
print(f"→ Wind < 1000 MW : {(gen < 1_000).mean()*100:.1f}% of all slots")
print(f"→ CoV {gen.std()/gen.mean()*100:.0f}% — extremely high variability "
      f"(gas turbine ~5–10%)")

   ACTUAL WIND GENERATION — DESCRIPTIVE STATISTICS
   N (30-min slots)          19
   Days of data              0.4
   Min                            5,126 MW
   P1                             5,174 MW
   P5                             5,364 MW
   P10  ← firm capacity           5,516 MW
   P20                            5,600 MW
   P25                            5,690 MW
   Median (P50)                   6,139 MW
   Mean                           6,772 MW
   P75                            7,513 MW
   P90                            9,045 MW
   P95                            9,587 MW
   Max                           10,248 MW
   Std Dev                        1,500 MW
   Coeff of Variation             22.1%

→ Wind < 500 MW  : 0.0% of all slots
→ Wind < 1000 MW : 0.0% of all slots
→ CoV 22% — extremely high variability (gas turbine ~5–10%)


In [14]:
df_daily = (
    df.set_index("startTime")
    .resample("D")["generation"]
    .agg(["min", "mean", "max"])
    .reset_index()
)

In [15]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.fill_between(df_daily["startTime"],
                df_daily["min"], df_daily["max"],
                alpha=0.20, color="#58a6ff", label="Daily min–max range")
ax.plot(df_daily["startTime"], df_daily["mean"],
        color="#58a6ff", lw=1.5, label="Daily mean")

In [16]:
for q, c, lbl in [
    (0.10, "#e84749", "P10  (firm capacity)"),
    (0.20, "#f0883e", "P20"),
    (0.50, "#3fb950", "Median"),
]:
    val = gen.quantile(q)
    ax.axhline(val, color=c, lw=1.8, ls="--",
               label=f"{lbl}: {val:,.0f} MW")

In [17]:
ax.set_xlabel("Date")
ax.set_ylabel("Wind Generation  [MW]")
ax.set_title("UK National Wind Power — Actual Generation  (Jan 2025 →)",
             fontsize=12, fontweight="bold", color="#e6edf3")
ax.legend(fontsize=9, ncol=3)
ax.grid(True)
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

In [18]:
plt.tight_layout()
plt.savefig("generation_timeseries.png", bbox_inches="tight", dpi=150)
plt.show()

In [19]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("UK Wind Generation — Distribution & Exceedance Curve",
             fontsize=13, fontweight="bold", color="#e6edf3", y=1.02)

Text(0.5, 1.02, 'UK Wind Generation — Distribution & Exceedance Curve')

In [20]:
ax1.hist(gen, bins=80, color="#58a6ff", alpha=0.50,
         edgecolor="none", density=True)
kde = gaussian_kde(gen, bw_method=0.08)
xs  = np.linspace(gen.min(), gen.max(), 500)
ax1.plot(xs, kde(xs), color="#79c0ff", lw=2.5, label="KDE")

In [21]:
for lbl, q, c in [
    ("P5",  0.05, "#e84749"),
    ("P10", 0.10, "#f0883e"),
    ("P20", 0.20, "#d29922"),
    ("P50", 0.50, "#3fb950"),
]:
    val = gen.quantile(q)
    ax1.axvline(val, color=c, lw=2.0, ls="--",
                label=f"{lbl}: {val:,.0f} MW")

In [22]:
ax1.set_xlabel("Wind Generation  [MW]")
ax1.set_ylabel("Density")
ax1.set_title("Empirical Distribution", fontsize=11)
ax1.legend(fontsize=8)
ax1.grid(True, axis="y")
ax1.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

In [23]:
sorted_gen = np.sort(gen.values)
exceedance = (1 - np.arange(1, len(sorted_gen)+1) / len(sorted_gen)) * 100

In [24]:
ax2.plot(sorted_gen, exceedance,
         color="#58a6ff", lw=2.5, label="All-hours exceedance")

In [25]:
for exc, col in [
    (90, "#e84749"), (80, "#f0883e"),
    (70, "#d29922"), (50, "#3fb950"),
]:
    val = gen.quantile(1 - exc/100)
    ax2.axhline(exc, color=col, lw=1.0, ls=":")
    ax2.axvline(val, color=col, lw=1.0, ls=":")
    ax2.annotate(
        f"{exc}%: {val:,.0f} MW",
        xy=(val, exc),
        xytext=(val + 150, exc + 2),
        color=col, fontsize=8,
        arrowprops=dict(arrowstyle="->", color=col, lw=0.8),
    )

In [26]:
ax2.set_xlabel("Wind Generation  [MW]")
ax2.set_ylabel("% of time  ≥  x")
ax2.set_title("Exceedance Probability Curve", fontsize=11)
ax2.set_ylim(0, 100)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())
ax2.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax2.grid(True)

In [27]:
plt.tight_layout()
plt.savefig("distribution_cdf.png", bbox_inches="tight", dpi=150)
plt.show()

In [28]:
print("Reliability Table:")
print("-" * 45)
for exc in [0.99, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.50]:
    val = gen.quantile(1 - exc)
    flag = " ← recommended" if exc == 0.90 else ""
    print(f"  {exc*100:.0f}%  P{(1-exc)*100:.0f}   {val:>8,.0f} MW{flag}")

Reliability Table:
---------------------------------------------
  99%  P1      5,174 MW
  95%  P5      5,364 MW
  90%  P10      5,516 MW ← recommended
  85%  P15      5,575 MW
  80%  P20      5,600 MW
  75%  P25      5,690 MW
  70%  P30      5,792 MW
  50%  P50      6,139 MW


In [29]:
df["hour"]  = df["startTime"].dt.hour
mask_peak   = df["hour"].isin(PEAK_HOURS)
gen_peak    = df.loc[mask_peak,  "generation"]
gen_offpeak = df.loc[~mask_peak, "generation"]

In [30]:
p10_all     = gen.quantile(0.10)
p10_peak    = gen_peak.quantile(0.10)
p10_offpeak = gen_offpeak.quantile(0.10)
p05_all     = gen.quantile(0.05)
p05_peak    = gen_peak.quantile(0.05)
p20_all     = gen.quantile(0.20)
p20_peak    = gen_peak.quantile(0.20)
p50_all     = gen.quantile(0.50)

In [31]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.hist(gen_offpeak, bins=70, color="#58a6ff", alpha=0.45,
        density=True, edgecolor="none",
        label="Off-peak hours")
ax.hist(gen_peak, bins=70, color="#f0883e", alpha=0.55,
        density=True, edgecolor="none",
        label=f"Peak hours  ({PEAK_HOURS[0]}–{PEAK_HOURS[-1]}:00 UTC)")

(array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan]),
 array([0.        , 0.01428571, 0.02857143, 0.04285714, 0.05714286,
        0.07142857, 0.08571429, 0.1       , 0.11428571, 0.12857143,
        0.14285714, 0.15714286, 0.17142857, 0.18571429, 0.2       ,
        0.21428571, 0.22857143, 0.24285714, 0.25714286, 0.27142857,
        0.28571429, 0.3       , 0.31428571, 0.32857143, 0.34285714,
        0.35714286, 0.37142857, 0.38571429, 0.4       , 0.41428571,
        0.42857143, 0.44285714, 0.45714286, 0.47142857, 0.48571429,
        0.5       , 0.51428571, 0.52857143, 0.54285714, 0.55714286,
        0.57142857, 0.58571429, 0.6       , 0.61428571, 

In [32]:
ax.axvline(p10_all,     color="#ffffff", lw=2.0, ls="--",
           label=f"P10 all-hours:  {p10_all:,.0f} MW")
ax.axvline(p10_peak,    color="#f0883e", lw=2.0, ls="-",
           label=f"P10 peak-hours: {p10_peak:,.0f} MW  ★")
ax.axvline(p10_offpeak, color="#58a6ff", lw=2.0, ls="-",
           label=f"P10 off-peak:   {p10_offpeak:,.0f} MW")

In [33]:
ax.set_xlabel("Wind Generation  [MW]")
ax.set_ylabel("Density")
ax.set_title("Wind Generation Distribution — Peak vs Off-Peak Hours",
             fontsize=12, fontweight="bold", color="#e6edf3")
ax.legend(fontsize=9)
ax.grid(True, axis="y")
ax.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

In [34]:
plt.tight_layout()
plt.savefig("peak_vs_offpeak.png", bbox_inches="tight", dpi=150)
plt.show()

In [35]:
diff = p10_peak - p10_all
print(f"\nP10 all-hours   : {p10_all:,.0f} MW")
print(f"P10 peak-hours  : {p10_peak:,.0f} MW  ({diff:+,.0f} MW vs all-hours)")
print(f"P10 off-peak    : {p10_offpeak:,.0f} MW")
level = "LOWER" if p10_peak < p10_all else "HIGHER OR SIMILAR"
print(f"\n→ Wind is {level} during peak hours.")


P10 all-hours   : 5,516 MW
P10 peak-hours  : nan MW  (+nan MW vs all-hours)
P10 off-peak    : 5,516 MW

→ Wind is HIGHER OR SIMILAR during peak hours.


In [36]:
hourly = (
    df.groupby("hour")["generation"]
    .agg(
        p10    = lambda x: x.quantile(0.10),
        p25    = lambda x: x.quantile(0.25),
        median = lambda x: x.quantile(0.50),
        p75    = lambda x: x.quantile(0.75),
        p90    = lambda x: x.quantile(0.90),
        mean   = "mean",
    )
    .reset_index()
)

In [37]:
fig, ax = plt.subplots(figsize=(13, 5))
h = hourly["hour"]
ax.fill_between(h, hourly["p10"], hourly["p90"],
                alpha=0.15, color="#58a6ff", label="P10–P90 band")
ax.fill_between(h, hourly["p25"], hourly["p75"],
                alpha=0.30, color="#58a6ff", label="IQR (P25–P75)")
ax.plot(h, hourly["median"], color="#58a6ff", lw=2.5, label="Median")
ax.plot(h, hourly["mean"],   color="#79c0ff", lw=1.5,
        ls="--", label="Mean")
ax.plot(h, hourly["p10"],    color="#e84749", lw=2.0,
        label="P10  (firm capacity)")
ax.axvspan(PEAK_HOURS[0], PEAK_HOURS[-1],
           alpha=0.08, color="#f0883e", label="Peak demand window")

In [38]:
ax.set_xlabel("Hour of Day  (UTC)")
ax.set_ylabel("Wind Generation  [MW]")
ax.set_title("Diurnal Generation Profile — P10, IQR, Median",
             fontsize=12, fontweight="bold", color="#e6edf3")
ax.set_xticks(range(0, 24))
ax.legend(fontsize=9, ncol=2)
ax.grid(True)
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

In [39]:
plt.tight_layout()
plt.savefig("diurnal_profile.png", bbox_inches="tight", dpi=150)
plt.show()

In [40]:
print("Hourly P10 (firm capacity by hour of day):")
print("-" * 50)
for _, row in hourly.iterrows():
    bar = "█" * max(1, int(row["p10"] / 250))
    print(f"  {int(row['hour']):02d}:00  {row['p10']:>6,.0f} MW  {bar}")

Hourly P10 (firm capacity by hour of day):
--------------------------------------------------
  00:00   9,587 MW  ██████████████████████████████████████
  01:00   8,345 MW  █████████████████████████████████
  02:00   7,299 MW  █████████████████████████████
  03:00   6,558 MW  ██████████████████████████
  04:00   6,165 MW  ████████████████████████
  05:00   5,858 MW  ███████████████████████
  06:00   5,552 MW  ██████████████████████
  07:00   5,152 MW  ████████████████████
  08:00   5,776 MW  ███████████████████████
  09:00   5,610 MW  ██████████████████████


In [41]:
df["month_key"] = df["startTime"].dt.to_period("M")

In [42]:
monthly_profile = (
    df.groupby("month_key")["generation"]
    .agg(
        p10    = lambda x: x.quantile(0.10),
        p20    = lambda x: x.quantile(0.20),
        median = lambda x: x.quantile(0.50),
        mean   = "mean",
        n      = "count",
    )
    .reset_index()
)
monthly_profile["month_str"] = monthly_profile["month_key"].astype(str)
monthly_profile = monthly_profile[monthly_profile["n"] >= 20 * 48]

In [43]:
x = range(len(monthly_profile))
w = 0.28
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar([i-w for i in x], monthly_profile["p10"],
       width=w, color="#e84749", alpha=0.85, label="P10")
ax.bar([i   for i in x], monthly_profile["median"],
       width=w, color="#58a6ff", alpha=0.85, label="Median")
ax.bar([i+w for i in x], monthly_profile["mean"],
       width=w, color="#3fb950", alpha=0.85, label="Mean")

<BarContainer object of 0 artists>

In [44]:
ax.set_xticks(list(x))
ax.set_xticklabels(monthly_profile["month_str"],
                   rotation=30, ha="right")
ax.set_ylabel("Wind Generation  [MW]")
ax.set_title("Monthly Generation Profile — P10, Median, Mean",
             fontsize=12, fontweight="bold", color="#e6edf3")
ax.legend(fontsize=9)
ax.grid(True, axis="y")
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

In [45]:
plt.tight_layout()
plt.savefig("monthly_profile.png", bbox_inches="tight", dpi=150)
plt.show()
print(monthly_profile[
    ["month_str","n","p10","p20","median","mean"]
].to_string(index=False, float_format=lambda v: f"{v:,.0f}"))

Empty DataFrame
Columns: [month_str, n, p10, p20, median, mean]
Index: []


In [46]:
df_ts = df.set_index("startTime").sort_index()
df_ts["roll_min_24h"] = (
    df_ts["generation"].rolling(48,  min_periods=36).min()
)
df_ts["roll_min_72h"] = (
    df_ts["generation"].rolling(144, min_periods=100).min()
)

In [47]:
DROUGHT_THR = 1_000 

In [48]:
fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(15, 9), sharex=True,
    gridspec_kw={"hspace": 0.08}
)
fig.suptitle("Wind Generation — Rolling Minimums & Drought Detection",
             fontsize=13, fontweight="bold", color="#e6edf3")

Text(0.5, 0.98, 'Wind Generation — Rolling Minimums & Drought Detection')

In [49]:
ax_top.fill_between(df_ts.index, 0, df_ts["generation"],
                    alpha=0.25, color="#58a6ff", label="Actual")
ax_top.plot(df_ts.index, df_ts["roll_min_24h"],
            color="#f0883e", lw=1.8, label="24 h rolling min")
ax_top.plot(df_ts.index, df_ts["roll_min_72h"],
            color="#e84749", lw=1.8, ls="--", label="72 h rolling min")
ax_top.axhline(DROUGHT_THR, color="#d29922", lw=1.2, ls=":",
               label=f"{DROUGHT_THR:,} MW drought threshold")
ax_top.set_ylabel("Wind Generation  [MW]")
ax_top.legend(fontsize=8, ncol=4)
ax_top.grid(True)
ax_top.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

In [50]:
is_drought = df_ts["roll_min_24h"] < DROUGHT_THR
ax_bot.fill_between(df_ts.index, 0, df_ts["generation"],
                    where=~is_drought,
                    alpha=0.30, color="#58a6ff", label="Normal")
ax_bot.fill_between(df_ts.index, 0, df_ts["generation"],
                    where=is_drought,
                    alpha=0.55, color="#e84749",
                    label=f"Drought: 24 h min < {DROUGHT_THR:,} MW")
ax_bot.axhline(DROUGHT_THR, color="#e84749", lw=1.5, ls="--")
ax_bot.set_xlabel("Date")
ax_bot.set_ylabel("Wind Generation  [MW]")
ax_bot.legend(fontsize=8)
ax_bot.grid(True)
ax_bot.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

In [51]:
plt.tight_layout()
plt.savefig("drought_analysis.png", bbox_inches="tight", dpi=150)
plt.show()

In [52]:
roll_24 = df_ts["roll_min_24h"].dropna()
roll_72 = df_ts["roll_min_72h"].dropna()
worst_24h     = roll_24.min() if len(roll_24) > 0 else np.nan
worst_72h     = roll_72.min() if len(roll_72) > 0 else np.nan
worst_24h_idx = roll_24.idxmin() if len(roll_24) > 0 else None

In [53]:
print(f"Worst 24 h rolling minimum : {worst_24h:,.0f} MW  (ending {worst_24h_idx})" if worst_24h_idx is not None else f"Worst 24 h rolling minimum : {worst_24h:,.0f} MW  (insufficient data for index)")
print(f"Worst 72 h rolling minimum : {worst_72h:,.0f} MW")
print()
for thr in [500, 1_000, 1_500, 2_000]:
    pct = (df_ts["roll_min_24h"] < thr).mean() * 100
    print(f"  % of time 24 h min < {thr:,} MW : {pct:.1f}%")

Worst 24 h rolling minimum : nan MW  (insufficient data for index)
Worst 72 h rolling minimum : nan MW

  % of time 24 h min < 500 MW : 0.0%
  % of time 24 h min < 1,000 MW : 0.0%
  % of time 24 h min < 1,500 MW : 0.0%
  % of time 24 h min < 2,000 MW : 0.0%


In [54]:
df["cf"] = df["generation"] / INSTALLED_MW

In [55]:
month_counts    = df.groupby("month_key").size()
complete_months = month_counts[month_counts >= 20*48].index
cf_monthly = (
    df[df["month_key"].isin(complete_months)]
    .groupby("month_key")["cf"]
    .agg(
        mean_cf   = "mean",
        p10_cf    = lambda x: x.quantile(0.10),
        median_cf = lambda x: x.quantile(0.50),
    )
    .reset_index()
)
cf_monthly["month_str"] = cf_monthly["month_key"].astype(str)

In [56]:
fig, ax = plt.subplots(figsize=(12, 4))
x = range(len(cf_monthly))
ax.plot(x, cf_monthly["mean_cf"]   * 100, color="#58a6ff",
        lw=2.5, marker="o", ms=5, label="Mean CF")
ax.plot(x, cf_monthly["median_cf"] * 100, color="#3fb950",
        lw=2.0, marker="s", ms=4, ls="--", label="Median CF")
ax.plot(x, cf_monthly["p10_cf"]    * 100, color="#e84749",
        lw=2.0, marker="^", ms=4, ls=":", label="P10 CF  (firm)")

In [57]:
ax.set_xticks(list(x))
ax.set_xticklabels(cf_monthly["month_str"], rotation=30, ha="right")
ax.set_ylabel("Capacity Factor  [%]")
ax.set_title(
    f"Monthly Capacity Factors  "
    f"(installed ≈ {INSTALLED_MW/1000:.0f} GW)",
    fontsize=12, fontweight="bold", color="#e6edf3",
)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(fontsize=9)
ax.grid(True)

In [58]:
plt.tight_layout()
plt.savefig("capacity_factor.png", bbox_inches="tight", dpi=150)
plt.show()

In [59]:
print(f"Overall mean CF : {df['cf'].mean()*100:.1f}%  "
      f"→ {df['cf'].mean()*INSTALLED_MW:,.0f} MW average")
print(f"Overall P10 CF  : {df['cf'].quantile(0.10)*100:.1f}%  "
      f"→ {df['cf'].quantile(0.10)*INSTALLED_MW:,.0f} MW firm")

Overall mean CF : 23.4%  → 6,772 MW average
Overall P10 CF  : 19.0%  → 5,516 MW firm


In [60]:
p10_by_month = monthly_profile.set_index("month_key")["p10"]
worst_month_key = p10_by_month.idxmin() if len(p10_by_month) > 0 else None
p10_worst_month = p10_by_month.min() if len(p10_by_month) > 0 else np.nan

In [61]:
estimates = [
    ("P1  all-hours",        gen.quantile(0.01), "Extreme — 1% risk"),
    ("P5  all-hours",        p05_all,             "Very conservative"),
    ("P5  peak-hours",       p05_peak,            "Conservative at peak"),
    ("P10 all-hours",        p10_all,             "Standard firm capacity"),
    ("P10 peak-hours ★",     p10_peak,            "RECOMMENDED"),
    ("P10 worst month",      p10_worst_month,     f"Worst month: {worst_month_key}"),
    ("P20 all-hours",        p20_all,             "Moderate reliability"),
    ("P20 peak-hours",       p20_peak,            "Moderate at peak"),
    ("Worst 24 h sustained", worst_24h,           "Extreme drought event"),
    ("Worst 72 h sustained", worst_72h,            "Multi-day drought floor"),
    ("Median (typical)",     p50_all,              "Average expectation"),
]

In [62]:
fig, ax = plt.subplots(figsize=(13, 8))

In [63]:
labels     = [e[0] for e in estimates]
values     = [e[1] for e in estimates]
bar_colors = [
    "#e84749" if "★" in e[0]
    else "#f0883e" if "P10" in e[0]
    else "#3fb950" if "Median" in e[0]
    else "#58a6ff"
    for e in estimates
]

In [64]:
bars = ax.barh(range(len(estimates)), values,
               color=bar_colors, alpha=0.85, height=0.65)

In [65]:
for bar, val in zip(bars, values):
    ax.text(
        val + 80, bar.get_y() + bar.get_height() / 2,
        f"{val:,.0f} MW",
        va="center", ha="left",
        color="#c9d1d9", fontsize=9, fontweight="bold",
    )

In [66]:
rec_idx = next(i for i, e in enumerate(estimates) if "★" in e[0])
bars[rec_idx].set_edgecolor("#ffffff")
bars[rec_idx].set_linewidth(2.0)

In [67]:
ax.set_yticks(range(len(estimates)))
ax.set_yticklabels(labels, fontsize=9.5)
ax.set_xlabel("Reliable Wind Generation  [MW]")
ax.set_title("Wind Power Reliability — All Methods Compared",
             fontsize=12, fontweight="bold", color="#e6edf3")
ax.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.grid(True, axis="x", alpha=0.4)

In [68]:
plt.tight_layout()
plt.savefig("reliability_summary.png", bbox_inches="tight", dpi=150)
plt.show()

In [69]:
print("Multi-Method Reliability Summary:")
print("─" * 68)
print(f"  {'Method':<28} {'MW':>9}    Note")
print("─" * 68)
for lbl, val, note in estimates:
    marker = "  ◄ RECOMMENDED" if "★" in lbl else ""
    print(f"  {lbl:<28} {val:>9,.0f}    {note}{marker}")
print("─" * 68)

Multi-Method Reliability Summary:
────────────────────────────────────────────────────────────────────
  Method                              MW    Note
────────────────────────────────────────────────────────────────────
  P1  all-hours                    5,174    Extreme — 1% risk
  P5  all-hours                    5,364    Very conservative
  P5  peak-hours                     nan    Conservative at peak
  P10 all-hours                    5,516    Standard firm capacity
  P10 peak-hours ★                   nan    RECOMMENDED  ◄ RECOMMENDED
  P10 worst month                    nan    Worst month: None
  P20 all-hours                    5,600    Moderate reliability
  P20 peak-hours                     nan    Moderate at peak
  Worst 24 h sustained               nan    Extreme drought event
  Worst 72 h sustained               nan    Multi-day drought floor
  Median (typical)                 6,139    Average expectation
──────────────────────────────────────────────────────────────────

In [70]:
fig, ax = plt.subplots(figsize=(13, 6))

In [71]:
sorted_all  = np.sort(gen.values)
exc_all     = (1 - np.arange(1, len(sorted_all)+1) / len(sorted_all)) * 100
ax.plot(sorted_all, exc_all,
        color="#58a6ff", lw=2.5, label="All-hours exceedance", zorder=3)

In [72]:
sorted_peak = np.sort(gen_peak.values)
exc_peak    = (1 - np.arange(1, len(sorted_peak)+1) / len(sorted_peak)) * 100
ax.plot(sorted_peak, exc_peak,
              color="#f0883e", lw=2.5, ls="--",
        label="Peak-hours exceedance", zorder=3)

In [73]:
ax.axvline(p10_peak,
           color="#e84749", lw=2.5, zorder=4,
           label=f"★ Recommended: {p10_peak:,.0f} MW  (P10 peak-hours)")
ax.axvline(p10_all,
           color="#d29922", lw=2.0, ls="--", zorder=4,
           label=f"P10 all-hours: {p10_all:,.0f} MW")
ax.axvline(p05_all,
           color="#8b949e", lw=1.5, ls=":", zorder=4,
           label=f"P5  all-hours: {p05_all:,.0f} MW")

In [74]:
ax.axhline(90, color="#30363d", lw=0.8, ls=":")
ax.text(200, 91, "90% exceedance threshold",
        color="#8b949e", fontsize=8)

Text(200, 91, '90% exceedance threshold')

In [75]:
ax.fill_betweenx(
    [0, 100], 0, p10_peak,
    alpha=0.07, color="#e84749",
    label="Firm capacity zone",
)

In [76]:
ax.set_xlabel("Wind Generation  [MW]")
ax.set_ylabel("% of time  ≥  x")
ax.set_title(
    "Final Reliability Recommendation — UK Wind Power",
    fontsize=13, fontweight="bold", color="#e6edf3",
)
ax.set_ylim(0, 100)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.legend(fontsize=9, loc="upper right")
ax.grid(True)

In [77]:
plt.tight_layout()
plt.savefig("final_recommendation.png", bbox_inches="tight", dpi=150)
plt.show()

In [78]:
derate_pct = p10_peak / INSTALLED_MW * 100
backup_needed = INSTALLED_MW - p10_peak

In [79]:
print()
print("╔══════════════════════════════════════════════════════════════════╗")
print("║         WIND POWER RELIABILITY — FINAL RECOMMENDATION           ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║                                                                  ║")
print(f"║   RECOMMENDED FIRM CAPACITY : {p10_peak:>8,.0f} MW                  ║")
print("║   Metric: P10 of actual generation during peak demand hours      ║")
print("║   (generation exceeded 90% of peak-demand 30-min slots)         ║")
print("║                                                                  ║")
print(f"║   De-rating vs ~{INSTALLED_MW/1000:.0f} GW installed : {derate_pct:.1f}%                   ║")
print("║                                                                  ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║   SUPPORTING EVIDENCE                                            ║")
print(f"║   P10 all-hours           : {p10_all:>8,.0f} MW                  ║")
print(f"║   P5  all-hours           : {p05_all:>8,.0f} MW  (stress-test)   ║")
print(f"║   P10 worst month ({str(worst_month_key):<7}): {p10_worst_month:>8,.0f} MW  (seasonal)   ║")
print(f"║   Worst 72 h sustained    : {worst_72h:>8,.0f} MW  (drought)      ║")
print(f"║   Median (typical output) : {p50_all:>8,.0f} MW                  ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║   REASONING                                                      ║")
print("║                                                                  ║")
print("║   1. WHY P10 AND NOT THE MINIMUM?                                ║")
print("║      The absolute minimum represents a once-in-season extreme.   ║")
print("║      Planning to this level requires impractical backup volume.  ║")
print("║      No technology is planned to its absolute worst case.        ║")
print("║                                                                  ║")
print("║   2. WHY P10 AND NOT THE MEAN?                                   ║")
print("║      The mean (expected output) is exceeded only ~50% of the     ║")
print("║      time — not a reliability standard. In 50% of hours wind     ║")
print("║      is below the mean, meaning demand cannot be reliably met.   ║")
print("║                                                                  ║")
print("║   3. WHY PEAK HOURS SPECIFICALLY?                                ║")
print("║      Security of supply matters most at 16:00–20:00 UTC when     ║")
print("║      electricity demand peaks. Using all-hours P10 would         ║")
print("║      overstate reliability if wind is lower at those hours.      ║")
print("║                                                                  ║")
print("║   4. ALIGNMENT WITH INDUSTRY PRACTICE                           ║")
print("║      GB Capacity Market de-rating factors (National Grid ESO     ║")
print("║      Electricity Capacity Report) use a similar exceedance-      ║")
print("║      based methodology at ~P10 for intermittent generators.      ║")
print("║                                                                  ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║   PLANNING IMPLICATION                                           ║")
print("║                                                                  ║")
print(f"║   Non-wind firm capacity needed during worst 10% of evenings:   ║")
print(f"║   {INSTALLED_MW/1000:.0f} GW installed  −  {p10_peak/1000:.1f} GW firm  =  "
      f"{backup_needed/1000:.1f} GW backup         ║")
print("║                                                                  ║")
print("║   This gap must be covered by gas peakers, interconnectors,      ║")
print("║   battery storage, or demand response.                           ║")
print("║                                                                  ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║   CAVEATS & LIMITATIONS                                          ║")
print("║                                                                  ║")
print("║   1. Dataset covers Jan 2025 onwards — winter-biased.            ║")
print("║      UK wind peaks in winter. When summer months are included    ║")
print("║      the P10 figure will likely fall. A full 12-month dataset    ║")
print("║      is needed for a robust annual firm capacity figure.         ║")
print("║                                                                  ║")
print("║   2. Installed capacity growth. New offshore wind farms coming   ║")
print("║      online will raise both the absolute MW and the P10 figure.  ║")
print("║      The de-rating % may remain similar even as MW grows.        ║")
print("║                                                                  ║")
print("║   3. Correlation with demand not modelled here. The most         ║")
print("║      dangerous scenario — simultaneous low wind + high demand    ║")
print("║      (cold, still anticyclone) — requires joint probability      ║")
print("║      analysis beyond the scope of this univariate study.         ║")
print("║                                                                  ║")
print("║   4. P10 is a statistical threshold, not a guarantee.            ║")
print("║      In any given year, wind could spend more or less than       ║")
print("║      10% of peak hours below this level depending on weather.    ║")
print("║                                                                  ║")
print("╚══════════════════════════════════════════════════════════════════╝")


╔══════════════════════════════════════════════════════════════════╗
║         WIND POWER RELIABILITY — FINAL RECOMMENDATION           ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║   RECOMMENDED FIRM CAPACITY :      nan MW                  ║
║   Metric: P10 of actual generation during peak demand hours      ║
║   (generation exceeded 90% of peak-demand 30-min slots)         ║
║                                                                  ║
║   De-rating vs ~29 GW installed : nan%                   ║
║                                                                  ║
╠══════════════════════════════════════════════════════════════════╣
║   SUPPORTING EVIDENCE                                            ║
║   P10 all-hours           :    5,516 MW                  ║
║   P5  all-hours           :    5,364 MW  (stress-test)   ║
║   P10 worst month (None   ):      nan MW  (seasonal)   ║
║   Wo